# Systèmes embarqués temps réel

## Qu'est-ce qu'un Système embarqué ?

Un système embarqué est un dispositif électronique intégrant un processeur sur lequel s'exécute un logiciel spécifique appelé *firmware*. Ce dispositif est conçu pour répondre à un objectif précis et défini, mission dédiée à remplir. Il n'est pas d'usage général, ni polyvalent comme un ordinateur personnel PC qui peut exécuter une multitude de programmes différents en meme temps.

La plupart de ces systèmes sont conçus sur la base des microcontrôleurs, SoC (System On Chip). Et dans ce cours, nous allons utiliser un microcontrôleur, le Pico RP2350 fabriqué par la societé Raspberry Pi. Les microcontrôleurs sont omniprésents dans notre quotidien, ils sont cachés dans des objets et ils représentent environ 98 % des microprocesseurs fabriqués dans le monde. Aujourd'hui nous pouvons affirmer que la quasi-totalité (plus de 90-95 %) des appareils électriques modernes intègrent désormais un système embarqué. 

## Qu'est-ce qu'un Système embarqué temps réel ?


Un système embarqué est dit *temps réel* lorsqu'il exécute des tâches ou réagit à des événements extérieurs dans un délai garanti. Les résultats produits ne sont pas seulement corrects, mais ils sont fournis dans un délai requis. C'est-à-dire que le moment de la livraison des résultats est capital. C'est là l'objet du cours : nous allons apprendre comment construire des systèmes qui répondent à cette contrainte, les résultats doivent être **prévisibles dans le temps**. Tous les systèmes construits doivent garantir qu'une tâche se terminera toujours en moins de X microsecondes, maximale non négociable, c'est le **déterminisme temporel**.

Prenons l'exemple du déclenchement d'un airbag lors du choc d'un véhicule : si le signal arrive ne serait-ce que 50 millisecondes après l'impact, le système est inutile et les conséquences pour le conducteur peuvent être graves. D'une autre façon, on peut dire que si les résultats du système produits sont corrects mais arrivent en retard, le système a échoué, ce qui est équivalent à une défaillance.

Il existe 3 types de systèmes temps réel : 

- **Systèmes temps réel stricts (Hard Real-Time)** : Le non-respect du délais entraîne une catastophe ou une défaillance totale du système. Ici, WCET doit toujours inférieur à le deadline (l’échéance) dans toute les conditions.  
Exemple : comme précédement cité, le déclenchement d'un airbag ou le système de freinage d'un train, commande de vol d'un avion. Si le signal arrive avec centaines de millisecondes de retard, le système est inutile et les conséquences peuvent être fatales.  

    Les systèmes de traitement numérique du signal (DSP) sont généralement des systèmes temps réel stricts. Prenons l'exemple d'un signal analogique échantillonné à 8 kHz (pour préserver des fréquences jusqu'à 4 kHz). La période d'échantillonnage est de 125 µs. Cela signifie que le système dispose de 125 µs pour effectuer tout le traitement nécessaire avant l'arrivée de l'échantillon suivant. Si le système ne peut pas suivre ce rythme, il échoue. C'est une caractéristique typique d'un système temps réel strict.

- **Systèmes temps réel mous (Soft Real-Time)** : Un léger retard est tolérable sans conséquences graves, même si l'on cherche toujours à respecter les délais.  
Exemple : Un distributeur automatique de billets (ATM). Si l'affichage met deux secondes de plus, l'utilisateur attend, mais le service est finalement rendu.

- **Systèmes temps réel fermes (Firm Real-Time)** : Le non-respect du délais rend le résultat inutile et rejeté, cependant cela ne détruit pas le système et ne provoque pas de défaillance catastrophique.  
Exemple : Un flux vidéo en direct, un système de contrôle qualité sur une ligne de production. Si une image est traitée trop tard, on l'ignore et on passe à la suivante, mais la qualité globale diminue. Ceci affecte directement la qualité du service du système.

Remarque : les frontières entre ces types peuvent être floues, mais elles guident le choix des mécanismes de conception. 

### Pourquoi les contraintes temporelles existent ?

Le système doit impérativement suivre la vitesse du monde réel, car l'environnement physique est par nature réactif et dynamique.
- **Le temps ne s'arrête pas** : Le monde extérieur évolue en permanence, même pendant que le processeur effectue ses calculs.
- **La physique est continue** : Pendant que le processeur "réfléchit", le moteur continue de tourner, le drone subit la gravité, le signal audio défile et la voiture avance.
- **Maîtrise du contrôle** : Si le temps de réponse du système est trop lent par rapport à la dynamique du phénomène physique, le contrôle est perdu, ce qui peut mener à une instabilité ou à un accident.

Prenons le cas d'un robot auto-équilibré : son contrôleur doit lire les données du capteur IMU, estimer l'angle via un filtre de Kalman, calculer la correction PID, puis commander les moteurs. Ce cycle doit s'exécuter avec précision plusieurs centaines de fois par seconde ; au moindre retard, le robot perd l'équilibre et chute.

*Un système temps réel n'est pas un système rapide, ni celui qui fonctionne correctement, mais c'est un système qui est capable de réagir correctement au bon moment.*
Dans le système classique qui n'est pas temps réel, le résultat seul est important ; par contre, dans le système temps réel, le résultat **ET** le temps de réponse sont critiques. Cette définition est au cœur de tout ce cours.

## Conception des systèmes embarqué temps réel

Ils sont sujets à des contraintes et des ressources :

- Dans ces systèmes temps réel, il n'y a pas de surprise, tout est déterministe.
- La taille de la mémoire est petite, de quelques kilo-octets à quelques méga-octets. Le choix des variables et la gestion de la mémoire font partie de l'ingénierie. Le code doit être extrêmement optimisé.
- Ils doivent consommer moins d'énergie électrique pour la plupart de ceux qui sont autonomes et qui doivent fonctionner sur des batteries.

Afin de parvenir à rendre un système "temps réel", nous allons mettre en œuvre les stratégies suivantes :

1. **Gestion des interruptions et Machines d'États (FSM)** : Utilisation conjointe d'interruptions et de machines d'états pour répondre aux tâches urgentes et prioritaires de manière structurée.
2. **ISR (Interrupt Service Routines) minimalistes** : Seules les actions critiques (capture d'un signal, arrêt d'urgence d'un moteur) seront traitées dans l'ISR. Tout traitement lourd est proscrit : l'ISR doit être la plus courte possible, se limitant généralement à lever un drapeau (flag) ou lire un registre.
3. **Traitement asynchrone** : Le gros des calculs est déporté dans la boucle principale (main), qui s'exécutera après la détection d'un flag levé par une interruption.
4. **Bannissement des fonctions bloquantes** : Évitement total des fonctions de type delay(). À la place, nous utiliserons des timers matériels cadencés qui lèvent des drapeaux à intervalles réguliers pour déclencher des actions ou simuler des délais sans figer le processeur.
5. **Exécution non-bloquante** : Chaque fonction doit s'exécuter très rapidement afin de "rendre la main" immédiatement au système et ne pas retarder les autres tâches.
6. **Utilisation du DMA (Direct Memory Access)** : Transfert de gros volumes de données entre la mémoire et les périphériques (ex: de l'ADC vers la RAM) sans solliciter le processeur. Ce dernier se contente d'ordonner le transfert et reçoit une notification une fois l'opération terminée.
7. **Exploitation du PIO (Programmable Input Output)** : Utilisation de ce coprocesseur spécialisé (similaire à un mini-FPGA intégré) pour créer des protocoles de communication personnalisés. Ils fonctionnent de manière autonome, libérant ainsi du temps de calcul précieux sur le processeur central.
8. **Analyse du WCET (Worst Case Execution Time)** : Calcul manuel du "Pire Cas" pour chaque chemin d'exécution du code. Si la somme des temps d'exécution dépasse l'échéance impartie, le système perd sa garantie temps réel.

Exemple concret : Un contrôleur de vitesse de moteur
1. Timer Interruption (chaque 500µs) : Lecture du capteur de position et calcul de la nouvelle commande (priorité absolue).
2. Boucle Main (Background) : Envoi des données vers un écran ou une console. Si l'envoi prend du temps, ce n'est pas grave, car l'interruption du Timer viendra "couper" ce travail pour assurer la régulation du moteur à temps.

Lorsque le système devient complexe, intégrant plusieurs tâches simultanées, des priorités variées, des interfaces de communication et des contraintes temporelles strictes; l’usage d’un **RTOS (Real-Time Operating System)** devient indispensable.   

Le RTOS est une couche logicielle supérieure (framework) qui permet d'organiser le code afin de :
- Garantir le respect des échéances (deadlines) du temps réel.
- Assurer la synchronisation entre les tâches.
- Gérer le partage du temps processeur (ordonnancement).  

On retrouve parmi les plus célèbres : FreeRTOS, VxWorks, Zephyr, QNX ou encore RTLinux.  

**Exemple concret d'un robot auto-équilibré**  
Supposons un système gérant trois tâches aux périodicités différentes : Lecture IMU toutes les 1 ms (Critique), la mise à jour Wi-Fi toutes les 50 ms (Lourde). Et l'affichage toutes les 20 ms (Secondaire).  

- Sans RTOS (Bare Metal), si la tâche Wi-Fi mobilise le processeur pendant 50 ms pour traiter des données, elle bloque l'exécution des autres fonctions. La lecture de l'IMU devra attendre la fin du traitement Wi-Fi. Pour un robot équilibré, ce retard est fatal, il ne peut plus corriger sa position et chute.  

- Avec RTOS, grâce à l'ordonnancement préemptif, le RTOS va suspendre immédiatement la tâche Wi-Fi dès que l'échéance de 1 ms de l'IMU arrive. L'IMU est traitée en priorité absolue, puis le processeur reprend le Wi-Fi là où il s'était arrêté.  

Un RTOS n'est pas conçu pour rendre un système plus rapide, mais pour rendre son comportement temporel déterministe et prévisible.

### Quelques concepts de temps reel

- **Deadline (Échéance)**: C'est le temps maximal (ou échéance maximale) autorisé pour terminer une tâche.  
Si un capteur finit sa lecture apres le temps predefini, il y a erreur temps reel.
- **Latence (délais d’attente)**: C'est le temps entre l’apparition d’un événement externe déclencheur et moment de la réaction du système (le début de son traitement).
Exemple : Un appui sur le bouton poussoir allume la LED. Si le délai trop grand, le système est peu réactif. Si on a 1ms c'est excellent, mais si on a 500 ms, c'est désagréable a l'utilisateur.
- **Temps de réponse** : est bien égal a la latence plus le temps d’exécution (le traitement). Plus il est faible, plus le système est réactif.
- **Jitter (gigue)** : est la variation imprévisible du temps d’exécution (latence).
Exemple : Un traitement audio doit durer : 22 µs - 22 µs - 22 µs et pas : 10 µs - 50 µs - 5 µs   
Le jitter crée de l’instabilité. Une forte gigue dégrade la précision temporelle. Le jitter est souvent plus dangereux que la latence moyenne.
- **Temps d'exécution WCET (Worst Case Execution Time)**: c’est la durée maximale que la tâche peut mettre pour terminer un travail, en considérant le pire chemin d’exécution (branchements, boucles, accès mémoire) et en l’absence d’interférences (interruptions, préemptions).
C'est ce temps qui intéresse l'ingénieur temps réel et pas le temps moyen.